Figure to compare the J0212 spectrum to the K dwarf and Giant

To that end, I'm going to pull code from "multiple_object_spectra.ipynb" since this is basically the same thing, but I don't want to try to clean up that whole mess.

In [1]:
from __future__ import print_function



import matplotlib
matplotlib.use('pdf')



import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

sys.path.append('../')



import spec_plot_tools as spt
import cal_params as cp

import plot_spec as ps

print(os.getcwd())

plt.rc('lines',linewidth=0.5)

filename: -f
/Users/BenKaiser/Desktop/radial_velocity_calculations/Kdwarf_impostor_plot_code


In [2]:
target_dir='/Users/BenKaiser/Desktop/Kdwarf_impostor_paper'
target_spec_file='ravg_fwctb.WDJ0212m5522_20231216_400m1.fits'
target_spec_file2='ravg_fwctb.WDJ0212m5522_20240111_400m2.fits'
dwarf_spec_file='K5_-2.0_Dwarf.fits'
giant_spec_file='K5_-2.0_Giant.fits'
#There isn't a 400M1 file for J2317 because we never took one in an attempt to maximize SNR in 400M2

In [3]:
os.chdir(target_dir)

In [4]:
figure_output_dir='/Users/BenKaiser/Desktop/Kdwarf_impostor_paper/figures/'

In [5]:
range_400m1=[3600,6660]
range_400m2=[6660,10000]

sdss_pix_bin=20.6

In [6]:
target_spec, header, target_noise= spt.retrieve_spec(target_spec_file)
target_spec2,header2, target_noise2=spt.retrieve_spec(target_spec_file2)
dwarf_spec, dwarf_header, dwarf_noise=spt.retrieve_sdss_spec(dwarf_spec_file)
giant_spec, giant_header, giant_noise=spt.retrieve_sdss_spec(giant_spec_file)

wavelengths.min:  3650.0496
new_wavelengths.min(): 3649.0096496361098
wavelengths.min:  3650.0496
new_wavelengths.min(): 3649.0096496361098


In [7]:
sm_target_spec=ps.convolve_spectrum(target_spec, header,kernel_type='box',pix_width=3)
sm_target_spec2=ps.convolve_spectrum(target_spec2, header2,kernel_type='box',pix_width=3)
#sm_dwarf_spec=ps.convolve_spectrum(dwarf_spec,header, kernel_type='box',pix_width=sdss_pix_bin)
#sm_giant_spec=ps.convolve_spectrum(giant_spec,header, kernel_type='box',pix_width=sdss_pix_bin)
sm_dwarf_spec=ps.convolve_spectrum(dwarf_spec,header, kernel_type='sdss_match',pix_width=header['see_sig'])
sm_giant_spec=ps.convolve_spectrum(giant_spec,header, kernel_type='sdss_match',pix_width=header['see_sig'])

trim_spot=6800




sdss_see_sig 0.9907997169143666
see_sig 2.276680050167974
conv_see_sig 2.0497775420262707
sdss_see_sig 0.9907997169143666
see_sig 2.276680050167974
conv_see_sig 2.0497775420262707


In [8]:
norm_range=[trim_spot-20., trim_spot+20.]

In [9]:
nsm_target_spec=ps.norm_spectrum(sm_target_spec,norm_range=norm_range)
nsm_target_spec2=ps.norm_spectrum(sm_target_spec2,norm_range=norm_range)
nsm_dwarf_spec=ps.norm_spectrum(sm_dwarf_spec,norm_range=norm_range)
nsm_giant_spec=ps.norm_spectrum(sm_giant_spec,norm_range=norm_range)



wavelength-based norm range selected [6780.0, 6820.0]





wavelength-based norm range selected [6780.0, 6820.0]





wavelength-based norm range selected [6780.0, 6820.0]





wavelength-based norm range selected [6780.0, 6820.0]





In [10]:
tsm_target_spec=spt.clean_spectrum(nsm_target_spec,np.min(target_spec[0]),trim_spot,[])
tsm_target_spec2=spt.clean_spectrum(nsm_target_spec2,trim_spot, np.max(target_spec2[0]),[])
tsm_dwarf_spec=spt.clean_spectrum(nsm_dwarf_spec,np.min(target_spec[0]),np.max(target_spec2[0]),[])
tsm_giant_spec=spt.clean_spectrum(nsm_giant_spec,np.min(target_spec[0]),np.max(target_spec2[0]),[])







In [13]:
#plt.rc('font',size=12)
#spt.initiate_science_plot()
#fig= plt.figure(figsize=(10,6))
fig=spt.start_ApJ_fig(width_cols=2,width_height=[1.,6./10.])
#spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='cool_wd',convert_to_air=True)

label_pos=3.3
label_off=20
#x_pos= 6000
#x_pos= 8000
x_pos= 6750
y_pos= 1.05
offset=0.3

plt.plot(tsm_target_spec[0], tsm_target_spec[1], color='k')
plt.plot(tsm_target_spec2[0], tsm_target_spec2[1], color='k')

plt.plot(tsm_dwarf_spec[0], tsm_dwarf_spec[1]+offset, color='k')

plt.plot(tsm_giant_spec[0], tsm_giant_spec[1]+offset*2, color='k')

plt.text(x_pos, y_pos, 'WD J0212–5522', color='k')
plt.text(x_pos, y_pos+offset, 'K5 Dwarf ([Fe/H]= –2.0)', color='k')
plt.text(x_pos, y_pos+offset*2, 'K5 Giant ([Fe/H]= –2.0)', color='k')

plt.ylabel(r'Flux ($f_{\lambda}$ arbitrary units)')
#plt.xlabel(r'$\lambda(\AA)$')
plt.xlabel(r'Wavelength $(\mathrm{\AA})$')



#legend_lines=['Li','Na','MgH','K','Ca']
#for element in legend_lines:
    #plt.axvline(x=-1000.,linestyle='--',color=cp.line_color_dict[element],label=element)

plt.xlim(3750,9000)
plt.ylim(-0.05,1.8)
#plt.legend(loc='lower right',framealpha=1)
#spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='cool_wd',convert_to_air=True)
#spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='h',show_telluric=False,convert_to_air=True)
spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='J0212',show_telluric=False,convert_to_air=True)









print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig('grav_effects_'+time_string+'.pdf')#plt.grid(True)



spt.show_plot(show_legend=False)

/Users/BenKaiser/Desktop/Goodman_ref_files/line_lists/J0212_comp_lines.csv
wavelengths.min:  5891.583264
new_wavelengths.min(): 5889.950866765008
wavelengths.min:  5897.558147
new_wavelengths.min(): 5895.924149766943
wavelengths.min:  3969.59
new_wavelengths.min(): 3968.4672118153667
wavelengths.min:  3934.77
new_wavelengths.min(): 3933.6562946887625
wavelengths.min:  4227.92
new_wavelengths.min(): 4226.729580953195
wavelengths.min:  8500.35
new_wavelengths.min(): 8498.015025790284
wavelengths.min:  8544.44
new_wavelengths.min(): 8542.09310564764
wavelengths.min:  8664.52
new_wavelengths.min(): 8662.140635748052
wavelengths.min:  5190.0
new_wavelengths.min(): 5188.554979826446
wavelengths.min:  4862.71
new_wavelengths.min(): 4861.351971801832
wavelengths.min:  6564.6
new_wavelengths.min(): 6562.787030044706
/Users/BenKaiser/Desktop/Kdwarf_impostor_paper/figures
/Users/BenKaiser/Desktop/Kdwarf_impostor_paper/figures
1716902156.690997


../spec_plot_tools.py:582: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [12]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,6))
plt.plot(norm_J1824_400m1[0], norm_J1824_400m1[1], color='k')
plt.plot(norm_J1824_400m2[0], norm_J1824_400m2[1], color='k')
plt.text(x_pos, y_pos, 'WD J1824+1213', color='k')
plt.ylabel(r'Flux ($f_{\lambda}$ arbitrary units)')
plt.xlabel(r'Wavelength $(\AA)$')
plt.xlim(3750,9000)


spt.show_plot(show_legend=False)


NameError: name 'norm_J1824_400m1' is not defined

In [ ]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,6))
plt.plot(norm_J1824_400m1[0], norm_J1824_400m1[1], color='k')
plt.plot(norm_J1824_400m2[0], norm_J1824_400m2[1], color='k')
plt.text(x_pos, y_pos, 'WD J1824+1213', color='k')
plt.ylabel(r'Flux ($f_{\lambda}$ arbitrary units)')
plt.xlabel(r'Wavelength $(\AA)$')
plt.xlim(7700,8000)


spt.show_plot(show_legend=False)